# Gestructureerde data: Complexere architecturen

In de voorgaande onderdelen heb je reeds gemerkt dat het verwerken van de input in het geval van gestructureerde data kan leiden tot een complexere netwerkarchitectuur.
In deze notebook gaan we dit in meer detail bestuderen en ook kijken naar de mogelijkheden aan de output-kant van het netwerk.

## Multi-modal modellen (meerdere inputs combineren voor 1 output)

Een multi-modal neuraal netwerk is een netwerk dat leert van meerdere soorten data (modaliteiten) tegelijk. Modaliteiten kunnen bijvoorbeeld zijn:

* Gestructureerde data (sensorwaarden, tabellen, tijdreeksen)
* Beelden (RGB-foto's, video frames)
* Tekst (reviews, captions, transcripties)
* Audio (spraak, muziek)

Het doel is om de informatie uit verschillende bronnen te combineren zodat het model betere prestaties kan leveren dan wanneer je slechts één modaliteit gebruikt.

Elke modaliteit heeft vaak een gespecialiseerde encoder:
| Modaliteit    | Veelgebruikte encoders                               |
| ------------- | ---------------------------------------------------- |
| Gestructureerde | Fully connected layers, tabular transformers         |
| Beeld         | CNN (ResNet, EfficientNet), ViT (Vision Transformer) |
| Tekst         | RNN, LSTM, GRU, Transformers (BERT, GPT)             |
| Audio         | 1D-CNN, spectrogram + 2D-CNN, Wav2Vec                |

Er zijn verschillende manieren om features van de verscheidene modaliteiten te combineren:

* Early fusion: De ruwe data wordt gecombineerd vóór het netwerk (vaak lastig bij verschillende soorten data).
* Intermediate fusion: Features van verschillende subnetwerken worden gecombineerd in een middenlaag (meest gebruikelijk).
* Late fusion: Elke modaliteit geeft een voorspelling en die worden gecombineerd (bijv. gemiddelde, gewogen gemiddelde, of meta-classifier). Dit lijkt op de klassieke ensemble technieken die we vorig jaar gezien hebben.

Met pytorch ziet dit eruit als volgt:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Encoder voor modaliteit 1 ---
class Modality1Encoder(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, out_dim=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.fc(x)

# --- Encoder voor modaliteit 2 ---
class Modality2Encoder(nn.Module):
    def __init__(self, input_dim=15, hidden_dim=32, out_dim=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.fc(x)

# --- Multi-modal model met feedforward subnetwerken ---
class MultiModalFFNN(nn.Module):
    def __init__(self, input_dim1, input_dim2, num_classes):
        super().__init__()
        self.mod1_encoder = Modality1Encoder(input_dim=input_dim1, out_dim=128)
        self.mod2_encoder = Modality2Encoder(input_dim=input_dim2, out_dim=128)
        self.classifier = nn.Linear(128*2, num_classes)
    
    def forward(self, x1, x2):
        feat1 = self.mod1_encoder(x1)
        feat2 = self.mod2_encoder(x2)
        combined = torch.cat([feat1, feat2], dim=1)  # feature fusion
        out = self.classifier(combined)
        return out

# --- Voorbeeld input ---
batch_size = 4
x1 = torch.randn(batch_size, 20)  # modaliteit 1
x2 = torch.randn(batch_size, 15)  # modaliteit 2

model = MultiModalFFNN(input_dim1=20, input_dim2=15, num_classes=5)
output = model(x1, x2)
print(output.shape)  # torch.Size([4, 5])


Met de functial API van Keras ziet dit er dan uit als volgt

## Multi-task/output/head model

Een multi-task neuraal netwerk is een netwerk dat meerdere gerelateerde taken tegelijk leert. In plaats van voor elke taak een afzonderlijk netwerk te trainen, wordt één netwerk gedeeld over taken. Dit kan leiden tot betere generalisatie, omdat het netwerk gedeelde representaties leert die nuttig zijn voor meerdere taken.

Voorbeelden van multi-task learning:

* Gezichtsherkenning: voorspellen van leeftijd, geslacht en emotie van hetzelfde gezicht.
* Tabulaire data: voorspellen van zowel een continu resultaat (regressie) als een categorie (classificatie) uit dezelfde features.
* Gezondheidszorg: voorspellen van meerdere symptomen of uitkomsten uit patiëntgegevens.

Een multi-task model heeft vaak gedeelde lagen (shared layers) die features leren die nuttig zijn voor alle taken, en taakspecifieke lagen (task-specific layers) die de uiteindelijke voorspelling voor elke taak maken. Dit resulteert inde volgende architectuur.

* De eerste lagen zijn gedeeld voor alle taken.
* Daarna heeft elke taak een eigen output layer.

Daarnaast moet er voldoende aandacht besteed worden aan de loss-functie van dit netwerk.
Omdat er nu meerdere taken zijn, zijn er ook meerdere fouten (1 per taak).
Om het netwerk correct te trainen moet je de loss-functies combineren tot een samengestelde lossfunctie (vermenigvuldig elke loss-functie met een factor die je kiest en tel ze op).
Het gewicht van de loss-functie is echter belangrijk voor goed learning en om een goede waarde te vinden kan het nodig zijn om met verschillende parameters te experimenteren.

Hieronder staat een eenvoudig feedforward multi-task model in PyTorch dat één regressie- en één classificatietaak tegelijk uitvoert.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Multi-task feedforward netwerk ---
class MultiTaskFFNN(nn.Module):
    def __init__(self, input_dim, shared_dim=64, task1_dim=32, task2_dim=32, num_classes=3):
        super().__init__()
        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, shared_dim),
            nn.ReLU()
        )
        
        # Taakspecifieke layers
        self.task1 = nn.Sequential(  # regressie
            nn.Linear(shared_dim, task1_dim),
            nn.ReLU(),
            nn.Linear(task1_dim, 1)  # output 1 waarde
        )
        
        self.task2 = nn.Sequential(  # classificatie
            nn.Linear(shared_dim, task2_dim),
            nn.ReLU(),
            nn.Linear(task2_dim, num_classes)  # logits
        )
    
    def forward(self, x):
        shared_feat = self.shared(x)
        out1 = self.task1(shared_feat)
        out2 = self.task2(shared_feat)
        return out1, out2

# --- Voorbeeld input ---
batch_size = 4
input_dim = 10
x = torch.randn(batch_size, input_dim)

model = MultiTaskFFNN(input_dim=input_dim, num_classes=3)
out1, out2 = model(x)
print("Regressie output:", out1.shape)  # torch.Size([4,1])
print("Classificatie output:", out2.shape)  # torch.Size([4,3])

# --- Voorbeeld training loop ---
y_reg = torch.randn(batch_size,1)
y_class = torch.randint(0,3,(batch_size,))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn_reg = nn.MSELoss()
loss_fn_class = nn.CrossEntropyLoss()

for epoch in range(5):
    optimizer.zero_grad()
    out1, out2 = model(x)
    loss1 = loss_fn_reg(out1, y_reg)
    loss2 = loss_fn_class(out2, y_class)
    loss = 0.4 * loss1 + 0.6 * loss2  # eenvoudige gewogen som
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

model_pytorch = model

Met Keras ziet dit eruit als volgt

## Visualisatie van model architecturen

Er zijn heel wat verschillende manieren om de netwerkarchitectuur te visualiseren om goed te begrijpen wat er gebeurd.
Hieronder staan er een aantal voorbeelden voor

**PYTORCH**

In [ ]:
# print model
print(model_pytorch)

In [ ]:
# print summary
from torchinfo import summary

summary(model_pytorch, input_size=(4,10))

In [ ]:
# print each layer
for name, layer in model.named_modules():
    print(name, layer)

In [ ]:
# make figure of the architecture
from torchviz import make_dot

x = torch.randn(1, 10, requires_grad=True)
x = x.to("cuda")
y1, y2 = model_pytorch(x)

combined = torch.cat([y1, y2.float()], dim=1)  # y2 naar float bij classificatie logits
dot = make_dot(combined, params=dict(model.named_parameters()))
dot.render("model_graph_combined", format="png") # kan verduidelijkt worden door:  with torch.autograd.profiler.record_function("shared") in de forward functie om stappen te combineren

**KERAS**

In [ ]:
model_keras.summary()

In [ ]:
for layer in model_keras.layers:
    print(layer)

In [ ]:
from keras_core.utils import plot_model

plot_model(model_keras, to_file="model.png", show_shapes=True, show_layer_names=True)

## Oefening - multitask

Ga aan de slag met [de wine-quality dataset](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009).
Je kan de dataset downloaden met kagglehub of manueel.

De op te lossen taken met deze dataset zijn:

* Taak 1: Voorspel de kwaliteit (quality) als integer score 0–10 → classificatie of regressie (kan als beiden)
* Taak 2: Voorspel of de wijn goed is of slecht (bijvoorbeeld quality >= 6) → binary classification

Voer hieronder de volgende stappen uit en train zowel een pytorch model als een keras model:

* Data inspecteren: bekijk statistieken, histogram van kwaliteit en good/bad labels.
* Preprocessing: normalisatie van features.
* Split train/test.
* Model bouwen:
  * Shared layers
  * Task-specific heads (regression + binary)
* Loss functies instellen (MSE voor regressie, BCE voor classificatie)
* Train model
* Evalueer prestaties op beide taken (MAE voor regressie, accuracy voor binary)

In [ ]:
# data voorbereiden

In [ ]:
# dataloaders

In [ ]:
# multi-task model

In [ ]:
# train the model

**Met Keras**

In [ ]:
# bouw en train model with keras